In [ ]:
!pip install spikingjelly
import torch
import torch.nn as nn
from spikingjelly.activation_based import neuron, functional, surrogate, layer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 13.5 MB/s eta 0:00:00


In [ ]:
class SimpleSNN(nn.Module):
    def __init__(self, tau=2.0):
        super().__init__()
        self.model = nn.Sequential(
            layer.Flatten(),
            layer.Linear(28 * 28, 128),
            neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan()),
            layer.Linear(128, 128),
            neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan()),
            layer.Linear(128, 10),
            neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan()),
        )

    def forward(self, x: torch.Tensor):
        return self.model(x)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.03MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 135kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.1MB/s]


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = SimpleSNN().to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

T = 6 # pasos de tiempo, cuanto mayor mas preciso es pero consume más
net.train()

for img, label in train_loader:
    img, label = img.to(device), label.to(device)
    optimizer.zero_grad()

    # hay que reiniciar la red antes de cada nueva imagen
    functional.reset_net(net)

    # procesar la imagen durante T pasos de tiempo
    out_spikes_counter = 0
    for t in range(T):
        out_spikes_counter += net(img)

    # Promedio de disparos (rate coding)
    output = out_spikes_counter / T
    loss = criterion(output, label)

    loss.backward()
    optimizer.step()

In [ ]:
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
correct = 0
total = 0

# modo evaluación
net.eval()

with torch.no_grad():
    for img, label in test_loader:
        img, label = img.to(device), label.to(device)

        # reiniciar siempre
        functional.reset_net(net)

        # sumar la actividad durante todos los pasos T
        out_spikes_counter = 0
        for t in range(T):
            out_spikes_counter += net(img)

        # la predicción es el índice con más actividad
        _, predicted = torch.max(out_spikes_counter, 1)

        # contar aciertos
        total += label.size(0)
        correct += (predicted == label).sum().item()

accuracy = 100 * correct / total
print(f'Precisión en el dataset de prueba: {accuracy:.2f}%')

Precisión en el dataset de prueba: 92.61%
